# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display, clear_output
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [4]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [5]:
#ed = Website("https://edwarddonner.com")
#print(ed.get_contents())
#ed.links

telman_eng = Website("https://www.telman-eng.co.il/")
print(telman_eng.get_contents())
telman_eng.links

['https://www.telman-eng.co.il',
 'https://www.telman-eng.co.il',
 'https://www.telman-eng.co.il/about',
 'https://www.telman-eng.co.il/services',
 'https://www.telman-eng.co.il/customers',
 'https://www.telman-eng.co.il/contact',
 'http://www.natix.co.il',
 'https://maps.google.co.il/maps?q=or+akiva+hahadas&hl=en&ll=32.516294,34.920666&spn=0.009047,0.021136&sll=32.510429,34.920768&sspn=0.00114,0.002642&gl=il&hnear=HaHadas,+Or+Akiva&t=m&z=16',
 'mailto:telman.yusupov@telman-eng.co.il']

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
# one-shot promt
#link_system_prompt = "You are provided with a list of links found on a webpage. \
#You are able to decide which of the links would be most relevant to include in a brochure about the company, \
#such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
#link_system_prompt += "You should respond in JSON as in this example:"
#link_system_prompt += """
#{
#    "links": [
#        {"type": "about page", "url": "https://full.url/goes/here/about"},
#        {"type": "careers page": "url": "https://another.full.url/careers"}
#    ]
#}
#"""

# multi-shot prompt
link_system_prompt = "You are provided with a list of links found on a webpage. \
    You are able to decide which of the links would be most relevant to include in a brochure about the company, \
    such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
    EXAMPLE 1:
    {
        "links": [
            {"type": "about page", "url": "https://full.url/goes/here/about"},
            {"type": "services page", "url": "https://another.full.url/services"}
        ]
    }
    EXAMPLE 2:
    {
        "links": [
            {"type": "customers page", "url": "https://full.url/goes/here/customers"},
            {"type": "contact page", "url": "https://full.url/goes/here/contact}
        ]
    }
    """

In [7]:
print(link_system_prompt)

You are provided with a list of links found on a webpage.     You are able to decide which of the links would be most relevant to include in a brochure about the company,     such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
    EXAMPLE 1:
    {
        "links": [
            {"type": "about page", "url": "https://full.url/goes/here/about"},
            {"type": "services page", "url": "https://another.full.url/services"}
        ]
    }
    EXAMPLE 2:
    {
        "links": [
            {"type": "customers page", "url": "https://full.url/goes/here/customers"},
            {"type": "contact page", "url": "https://full.url/goes/here/contact}
        ]
    }
    


In [8]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [9]:
print(get_links_user_prompt(telman_eng))

Here is the list of links on the website of https://www.telman-eng.co.il/ - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://www.telman-eng.co.il
https://www.telman-eng.co.il
https://www.telman-eng.co.il/about
https://www.telman-eng.co.il/services
https://www.telman-eng.co.il/customers
https://www.telman-eng.co.il/contact
http://www.natix.co.il
https://maps.google.co.il/maps?q=or+akiva+hahadas&hl=en&ll=32.516294,34.920666&spn=0.009047,0.021136&sll=32.510429,34.920768&sspn=0.00114,0.002642&gl=il&hnear=HaHadas,+Or+Akiva&t=m&z=16
mailto:telman.yusupov@telman-eng.co.il


In [11]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [12]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

telman_eng = Website("https://www.telman-eng.co.il/")
telman_eng.links

['https://www.telman-eng.co.il',
 'https://www.telman-eng.co.il',
 'https://www.telman-eng.co.il/about',
 'https://www.telman-eng.co.il/services',
 'https://www.telman-eng.co.il/customers',
 'https://www.telman-eng.co.il/contact',
 'http://www.natix.co.il',
 'https://maps.google.co.il/maps?q=or+akiva+hahadas&hl=en&ll=32.516294,34.920666&spn=0.009047,0.021136&sll=32.510429,34.920768&sspn=0.00114,0.002642&gl=il&hnear=HaHadas,+Or+Akiva&t=m&z=16',
 'mailto:telman.yusupov@telman-eng.co.il']

In [13]:
#get_links("https://www.anthropic.com/")
get_links("https://www.telman-eng.co.il/")

{'links': [{'type': 'about page', 'url': 'https://www.telman-eng.co.il/about'},
  {'type': 'services page', 'url': 'https://www.telman-eng.co.il/services'},
  {'type': 'customers page', 'url': 'https://www.telman-eng.co.il/customers'},
  {'type': 'contact page', 'url': 'https://www.telman-eng.co.il/contact'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [14]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [15]:
print(get_all_details("https://www.telman-eng.co.il/"))

Landing page:
Webpage Title:
TELMAN_ENG system design house
Webpage Contents:
top of page
HOME
ABOUT
SERVICES
CUSTOMERS
CONTACT
TELMAN-ENG Hardware Design House
TELMAN-ENG is a hardware design house for multidisciplinary system design with a focus on embedded systems on SOC/FPGA.
TELMAN-ENG is experienced with a wide range of industries from Novel Consumer Electronics and Industrial applications to Medical and Defense. Our company provides a full suite of design and development services with an emphases on close customer support and design flexibility.
TELMAN-ENG provides a complete solution of HW and SW design and development together with
NATIX SYSTEMS Ltd
software design house.
To play, press and hold the enter key. To stop, release the enter key.
Visit
﻿1 Hahadas Street
PO Box 254
Or Aqiva 3065201
Israel
﻿
http://goo.gl/maps/sEDPd
Call
Tel: 972-4-6101579
Fax: 972-4-6101647
Cell: 972-52-3764452
Contact
telman.yusupov@telman-eng.co.il
bottom of page



about page
Webpage Title:
ABOUT

In [16]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:20_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("Telman-Eng", "https://www.telman-eng.co.il/")

'You are looking at a company called: Telman-Eng\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nTELMAN_ENG system design house\nWebpage Contents:\ntop of page\nHOME\nABOUT\nSERVICES\nCUSTOMERS\nCONTACT\nTELMAN-ENG Hardware Design House\nTELMAN-ENG is\xa0a hardware\xa0design house for\xa0multidisciplinary\xa0system design with a focus on embedded systems on SOC/FPGA.\nTELMAN-ENG is experienced with a wide range of industries from Novel Consumer Electronics and Industrial applications to Medical and Defense. Our company provides a full suite of design and development services with an emphases on close\xa0customer\xa0support and design flexibility.\nTELMAN-ENG provides a complete solution of HW and SW design and development together with\nNATIX\xa0SYSTEMS\xa0Ltd\nsoftware design house.\nTo play, press and hold the enter key. To stop, release the enter key.\nVisi

In [19]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [20]:
create_brochure("Telman-Eng", "https://www.telman-eng.co.il/")
#create_brochure("HuggingFace", "https://huggingface.co")

# Telman-Eng Brochure

## About Us
**Telman-Eng** is a premier hardware design house established in 2008, specializing in multidisciplinary system design, particularly focused on embedded systems on SOC/FPGA. Our experienced team of hardware and software engineers has extensive backgrounds in the high-tech industry, positioning us as leaders in providing innovative solutions across various sectors including consumer electronics, medical, industrial, and defense applications.

### Our Services
We offer a comprehensive suite of design and development services:

- **FPGA Design**: Specialized implementation using XILINX families ranging from SPARTAN to ZYNQ.
- **Board Design**: Complete flow from concept to prototype with flexible client-driven designs.
- **Software Development**: End-to-end embedded software solutions on processors including ARM, Microblaze, and PowerPC.
- **RF Design**: Innovative wireless system solutions tailored to meet specific requirements.

With our strong project management capabilities, we ensure smooth progress and successful delivery of every project.

## Our Customers
We take pride in serving a diverse array of clients who trust us with their essential design needs:

- **IAI**: Embedded products including board designs and implementations on Xilinx FPGA.
- **EDGE Medical Devices**: Data acquisition board design for X-ray sensors.
- **Aspect Imaging**: System design and consultancy for MRI spectrometers and NMR spectroscopy.
- **Maradin**: MEMS projector control unit for real-time image correction based on XILINX ZYNQ FPGA.
- **Astronautics**: Design services encompassing simulation workspace development and logic verification.

## Company Culture
At Telman-Eng, we believe in fostering a collaborative and innovative environment. Our team is empowered to think creatively while maintaining a strong focus on customer support and satisfaction. We value flexibility in our designs, understanding that every client has unique needs. Our commitment to quality and teamwork drives our success and that of our partners.

## Careers
Join the **Telman-Eng** team! We are always looking for skilled individuals passionate about hardware and software design to contribute to our dynamic projects. If you are interested in being part of a driven team that values innovation and collaboration, please reach out to us.

### Contact Us
For more information, inquiries, or to learn about job opportunities, please contact us at:

- **Address**: 1 Hahadas Street, PO Box 254, Or Aqiva 3065201, Israel
- **Phone**: +972-4-6101579
- **Email**: [telman.yusupov@telman-eng.co.il](mailto:telman.yusupov@telman-eng.co.il)

**Together, let's engineer tomorrow's solutions!**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [21]:
def stream_brochure(company_name, url):
    global brochure_text  # Access the global variable
    brochure_text = ""    # Initialize
    
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        content = chunk.choices[0].delta.content or ''
        response += content
        brochure_text += content  # Accumulate the text
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
stream_brochure("Telman-Eng", "https://www.telman-eng.co.il/")

# Telman-Eng: Your Partner in System Design

Welcome to **Telman-Eng**, an innovative hardware design house specializing in multidisciplinary system design, with a focus on embedded systems utilizing SOC and FPGA technology. Established in **2008**, we have combined years of expertise from our professional hardware and software engineers to serve a diverse range of industries, including consumer electronics, medical, military, and industrial applications.

## Our Expertise

At Telman-Eng, we are dedicated to providing **End-to-End Solutions** covering every aspect of hardware and software design, ensuring flexibility and close collaboration with our clients. Our services encompass:
- **FPGA Design:** Proven expertise using Xilinx families from Spartan to Zynq.
- **Board Design:** Comprehensive board design services from concept to prototype.
- **Software Design:** Complete embedded software solutions on ARM, Microblaze, and Power PC processors, based on Java, Linux, Android, and Windows technologies.
- **RF Design:** Custom RF design and integrated wireless system solutions.

## Key Clients and Projects

We pride ourselves on our successful collaborations with leading companies, including:
- **IAI:** Development of various embedded products, including board design and embedded system implementations.
- **EDGE Medical Devices:** Data acquisition board design for X-Ray sensors.
- **Aspect Imaging:** System design and consulting for MRI spectrometers.
- **Maradin:** MEMS projector control unit development based on Xilinx Zynq FPGA.

## Our Culture

At Telman-Eng, our team is driven by a passion for innovation and a commitment to excellence. We foster a **collaborative and flexible culture**, encouraging our employees to continuously develop their skills and contribute towards meaningful projects. We believe in the importance of a supportive environment that values creativity and initiative.

## Careers at Telman-Eng

Join our dynamic team and be part of an organization where your contributions can lead to groundbreaking advancements. We are always on the lookout for passionate engineers and tech enthusiasts who share our commitment to quality and innovation. Whether you're a seasoned professional or a fresh recruit eager to learn, Telman-Eng offers a nurturing environment that helps shape your career.

## Get in Touch

We are here to assist you! Whether you have a project in mind, require our services, or want to learn more about career opportunities, please reach out to us:

**Contact Information:**
- **Phone:** +972-4-6101579
- **Fax:** +972-4-6101647
- **Cell:** +972-52-3764452
- **Email:** [telman.yusupov@telman-eng.co.il](mailto:telman.yusupov@telman-eng.co.il)
- **Address:** 1 Hahadas Street, PO Box 254, Or Aqiva 3065201, Israel

Explore how Telman-Eng can help bring your ideas to life! For more information, visit our website or contact us directly. We look forward to working together!

In [23]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Company Brochure

## Overview

### Hugging Face
**Website:** [huggingface.co](https://huggingface.co)  
**Tagline:** The AI community building the future.  
Hugging Face is a collaborative platform where the machine learning community comes together to create, discover, and discuss AI models, datasets, and applications. With access to over 2 million models and a strong focus on open-source contributions, Hugging Face is setting the stage for the future of machine learning.

---

## Company Culture

At Hugging Face, the culture is driven by community and open-source values. The team is on a mission to democratize machine learning, encouraging collaboration and innovation through shared knowledge and resources. With a strong focus on inclusivity, Hugging Face supports contributions from users worldwide, valuing every effort to advance the field of AI.

### Community Engagement
- **Active Contributions:** Notable events include users submitting papers, updating datasets, and collaborating on projects.
- **Learning & Growth:** Hugging Face emphasizes continuous learning with resources like blog articles, documentation, and community forums.

---

## Customers

Hugging Face serves a diverse range of over 50,000 organizations, including prominent companies such as:
- **Meta**
- **Microsoft**
- **Google**
- **Amazon**
- **OpenAI**

These organizations rely on Hugging Face for its state-of-the-art AI solutions, advanced research technology, and accessible collaboration tools.

---

## Products and Offerings

- **Models:** Explore a library of over 2 million machine learning models suitable for various tasks including text, image, and video processing.
- **Datasets:** Access and share over 500,000 datasets for any ML application.
- **Spaces:** Create interactive demos and applications using Hugging Face's tools.
- **Enterprise Solutions:** Tailored offerings for businesses, including enterprise-grade security, dedicated support, and advanced compute options.

### Pricing Plans
1. **PRO Account:** $9/month for enhanced personal features including private storage and priority support.
2. **Team Account:** $20/user/month for collaborative tools aimed at growing teams.
3. **Enterprise Account:** Custom solutions starting at $50/user/month with maximum capacity and support.

---

## Careers and Opportunities

Hugging Face is always on the lookout for passionate individuals to join their team. The company offers various roles in engineering, product management, and community engagement to foster innovation in the AI field.

### Work Environment
- **Dynamic Team:** Collaborate with over 187 team members and tap into a thriving community.
- **Innovative Projects:** Get involved with cutting-edge projects that impact the AI landscape.
- **Inclusive Culture:** Contribute to a workplace that values diverse perspectives and promotes growth.

---

Join Hugging Face in building the future of AI and become part of an inspiring community of innovators, researchers, and developers. Explore the endless possibilities in machine learning!

---

For more information, visit: [huggingface.co](https://huggingface.co)

In [24]:
def translate_brochure(lang):
    # Clear previous output
    clear_output(wait=True)
    
    # Stream #2: translate accumulated text
    translation_stream = openai.chat.completions.create(  # Changed from ChatCompletion
        model=MODEL,
        messages=[
            {"role": "user", "content": f"Translate the following to {lang}:\n\n{brochure_text}"}
        ],
        stream=True
    )
    
    # Setup display for streaming translation
    display_handle = display(Markdown(""), display_id=True)
    translated_text = ""
    
    for chunk in translation_stream:
        content = chunk.choices[0].delta.content or ""
        if content:
            translated_text += content
            update_display(Markdown(translated_text), display_id=display_handle.display_id)



In [27]:
# prompt user for language choice
language_choice = input("Enter the language to translate the brochure into (e.g., 'Russian'): ")

# translate the brochure and stream the translation
translate_brochure(language_choice)

# Брошюра компании Hugging Face

## Обзор

### Hugging Face
**Веб-сайт:** [huggingface.co](https://huggingface.co)  
**Слоган:** Сообщество ИИ, создающее будущее.  
Hugging Face — это совместная платформа, где сообщество машинного обучения собирается вместе для создания, открытия и обсуждения моделей ИИ, наборов данных и приложений. С доступом к более чем 2 миллионам моделей и сильным акцентом на вклад с открытым исходным кодом, Hugging Face подготавливает почву для будущего машинного обучения.

---

## Корпоративная культура

В Hugging Face культура основана на ценностях сообщества и открытого кода. Команда стремится демократизировать машинное обучение, поощряя сотрудничество и инновации через совместные знания и ресурсы. Обладая сильным акцентом на инклюзивность, Hugging Face поддерживает вклад пользователей по всему миру, ценя каждое усилие по развитию области ИИ.

### Вовлеченность сообщества
- **Активные вклады:** Заметные события включают представление пользователями статей, обновление наборов данных и сотрудничество по проектам.
- **Обучение и развитие:** Hugging Face подчеркивает непрерывное обучение с помощью ресурсов, таких как статьи в блоге, документация и форумы сообщества.

---

## Клиенты

Hugging Face обслуживает разнообразный круг из более чем 50,000 организаций, включая крупные компании, такие как:
- **Meta**
- **Microsoft**
- **Google**
- **Amazon**
- **OpenAI**

Эти организации полагаются на Hugging Face благодаря современным решениям в области ИИ, продвинутым исследовательским технологиям и доступным инструментам для совместной работы.

---

## Продукты и услуги

- **Модели:** Исследуйте библиотеку из более чем 2 миллионов моделей машинного обучения, подходящих для различных задач, включая обработку текста, изображений и видео.
- **Наборы данных:** Получайте доступ к более чем 500,000 наборов данных и делитесь ими для любых ML-приложений.
- **Пространства:** Создавайте интерактивные демонстрации и приложения, используя инструменты Hugging Face.
- **Корпоративные решения:** Индивидуальные предложения для бизнеса, включая корпоративную безопасность, специализированную поддержку и продвинутые вычислительные возможности.

### Тарифные планы
1. **PRO аккаунт:** $9/месяц за улучшенные персональные функции, включая частное хранилище и приоритетную поддержку.
2. **Командный аккаунт:** $20/пользователь/месяц за инструменты для совместной работы, ориентированные на растущие команды.
3. **Корпоративный аккаунт:** Индивидуальные решения начиная от $50/пользователь/месяц с максимальной мощностью и поддержкой.

---

## Карьера и возможности

Hugging Face всегда ищет увлеченных людей, желающих присоединиться к их команде. Компания предлагает различные роли в инженерии, управлении продуктами и вовлеченности сообщества, чтобы способствовать инновациям в области ИИ.

### Рабочая среда
- **Динамичная команда:** Сотрудничайте с более чем 187 членами команды и погружайтесь в процветающее сообщество.
- **Инновационные проекты:** Участвуйте в передовых проектах, которые влияют на ландшафт ИИ.
- **Инклюзивная культура:** Вносите свой вклад в рабочую среду, которая ценит разнообразные точки зрения и способствует росту.

---

Присоединяйтесь к Hugging Face в создании будущего ИИ и станьте частью вдохновляющего сообщества новаторов, исследователей и разработчиков. Исследуйте безграничные возможности в машинном обучении!

---

Для получения дополнительной информации, посетите: [huggingface.co](https://huggingface.co)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>